# 29. 묶음 코호트 쌍(KIPAN↔KIRC, GBMLGG↔LGG) 계층 분류 (H29)

## 배경
- 26개 라벨 중 2쌍은 서로 겹치는 코호트: **KIPAN ⊃ KIRC** (신장암 묶음), **GBMLGG ⊃ LGG** (뇌종양 묶음)
- 후보 C의 암종별 F1이 가장 낮은 쪽에 이 4개가 몰려 있음 (공통 CV: KIRC 0.19, KIPAN 0.23, GBMLGG 0.33, LGG 0.48)
- README 향후 개선 방향 4번: "묶음 코호트 구조를 고려한 계층형 분류 검토"

## 가설
- H29: 1단계에서 묶음(신장, 뇌)을 고르고 2단계에서 묶음 안의 세부 암종을 따로 고르는 계층 분류가 C보다 높다

## 검증 방법
- 1단계: 후보 C(22b XGB 0.7 + LR 0.3, 시드 42)의 확률에서 쌍의 확률을 합침 → 묶음 확률
- 2단계: 쌍에 속한 학습 환자만으로 "세부 암종형인지"를 맞히는 LR(any, C=0.1, balanced)
  - 학습 목표: 같은 변이 프로필 묶음 안에 세부 암종(KIRC, LGG) 라벨이 하나라도 있으면 1 (학습 fold 안에서만 계산)
  - 최종 확률: 묶음 확률 × 2단계 확률 (세부 암종), 묶음 확률 × (1 − 2단계 확률) (묶음 코호트)
- **검증을 두 가지로** 봄
  - ① 공통 CV (`get_folds`, 팀 기준)
  - ② 같은 변이 프로필을 한 fold에 묶은 CV (`StratifiedGroupKFold`) → train 안에 같은 프로필·다른 라벨 행이 많아서 ①이 왜곡되는지 확인용
- 채택 기준: C보다 5 fold 중 4개 이상 상승 (①, ② 모두에서 올라야 채택)
- **하지 않는 것** (팀 규칙): test 환자를 train의 같은 프로필 행과 맞춰 보는 것, 같은 프로필 행의 라벨로 예측을 고치는 후처리, 예측 비율 맞추기. 프로필 묶음은 **train 안에서만** 계산


In [1]:
import sys; sys.path.append('..')
import warnings; warnings.filterwarnings('ignore')
import time, platform, numpy as np, pandas as pd, xgboost as xgb
from collections import Counter
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED, WT
from src.data import load_data, get_gene_cols
from src.features import mutation_features, mutation_type, aa_substitution
from src.validation import seed_everything, encode_target, get_folds, macro_f1

print(f'{platform.system()} | CPU {platform.machine()} | xgboost {xgb.__version__}')
seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
ci = {c: i for i, c in enumerate(le.classes_)}
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

def sub_ratio_matrix(df):   # 22와 동일
    rows = []
    for row in df[genes].to_numpy():
        subs = (aa_substitution(t) for v in row if v != WT for t in v.split() if mutation_type(t) == 'missense')
        c = Counter(s for s in subs if s is not None)
        tot = sum(c.values())
        rows.append({k: n / tot for k, n in c.items()} if tot else {})
    M = pd.DataFrame(rows, index=df.index).fillna(0).astype(np.float32)
    M.columns = ['sub__' + k.replace('>', '_') for k in M.columns]
    return M

S_tr = sub_ratio_matrix(train)
S_te = sub_ratio_matrix(test).reindex(columns=S_tr.columns, fill_value=0)
X22b, X22b_te = pd.concat([F_tr['any'], F_tr['ratio'], S_tr], axis=1), pd.concat([F_te['any'], F_te['ratio'], S_te], axis=1)
A, A_te = F_tr['any'].to_numpy(np.float32), F_te['any'].to_numpy(np.float32)
PAIRS = [('KIPAN', 'KIRC'), ('GBMLGG', 'LGG')]   # (묶음 코호트, 그 안의 세부 암종)
FOUR = ['KIPAN', 'KIRC', 'GBMLGG', 'LGG']

Linux | CPU x86_64 | xgboost 3.4.1


#### 1) train 안의 같은 변이 프로필·다른 라벨 행
- 유전자 4,384칸이 모두 같은 행을 한 묶음으로 봄 (train 안에서만)

In [2]:
grp = pd.Series(pd.util.hash_pandas_object(train[genes], index=False).to_numpy()).rank(method='dense').astype(int).to_numpy()
g = pd.DataFrame({'g': grp, 'lab': train.SUBCLASS.values})
size = g.groupby('g').size(); multi = size[size > 1].index
print(f'train {len(train)}행 | 고유 프로필 {size.shape[0]}개 | 2행 이상 같은 프로필 묶음 {len(multi)}개 (행 {int(size[size > 1].sum())}개)')
combo = g[g.g.isin(multi)].groupby('g').lab.apply(lambda s: ' + '.join(sorted(set(s)))).value_counts()
print('묶음의 라벨 조합 (상위):'); print(combo.head(4).to_string())
for big, sub in PAIRS:
    for lab, other in [(big, sub), (sub, big)]:
        rows = np.where(y == ci[lab])[0]
        has = np.array([((grp == grp[r]) & (y == ci[other])).any() for r in rows])
        print(f'{lab}: {len(rows)}명 중 같은 프로필의 {other} 행이 있는 비율 {has.mean():.1%}')
FOLDS_STD = get_folds(y)
FOLDS_GRP = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y, grp))
fid = np.zeros(len(y), int)
for k, (_, va) in enumerate(FOLDS_STD): fid[va] = k
split = g.assign(f=fid)[g.g.isin(multi)].groupby('g').f.nunique()
print(f'공통 CV에서 같은 프로필 묶음이 서로 다른 fold로 갈라진 비율: {(split > 1).mean():.1%}')

train 6201행 | 고유 프로필 5636개 | 2행 이상 같은 프로필 묶음 451개 (행 1016개)
묶음의 라벨 조합 (상위):
lab
KIPAN + KIRC                                                                                                272
GBMLGG + LGG                                                                                                173
LAML                                                                                                          3
BRCA + CESC + GBMLGG + KIPAN + KIRC + LAML + LGG + LUSC + PAAD + PCPG + PRAD + SARC + STES + THCA + THYM      1
KIPAN: 515명 중 같은 프로필의 KIRC 행이 있는 비율 53.8%
KIRC: 334명 중 같은 프로필의 KIPAN 행이 있는 비율 82.6%
GBMLGG: 461명 중 같은 프로필의 LGG 행이 있는 비율 38.0%
LGG: 229명 중 같은 프로필의 GBMLGG 행이 있는 비율 76.9%


공통 CV에서 같은 프로필 묶음이 서로 다른 fold로 갈라진 비율: 80.7%


#### 2) 후보 C를 두 가지 fold로 학습
- 확률은 fold별로 `submissions/cache29/`에 저장 (git 제외, 중간에 끊겨도 이어서 실행). 캐시가 없으면 약 15분

In [3]:
CACHE = Path('../submissions/cache29'); CACHE.mkdir(parents=True, exist_ok=True)
K = len(le.classes_)

def run_c(folds, tag):
    """후보 C(22b XGB 0.7 + LR 0.3)의 OOF·test 확률. 치환 상위 100개는 fold 학습 부분에서만 선택 (22와 동일)"""
    oof, te = np.zeros((len(y), K)), np.zeros((len(test), K))
    for k, (tr, va) in enumerate(folds):
        p = CACHE / f'{tag}fold{k}.npz'
        z = np.load(p) if p.exists() else None
        if z is None or not np.array_equal(z['va'], va):
            t = time.time()
            sub = [c for c in X22b.columns if c.startswith('sub__')]
            keep = set(X22b.iloc[tr][sub].sum().sort_values(ascending=False).index[:100])
            drop = [c for c in sub if c not in keep]
            m = xgb.XGBClassifier(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.3, tree_method='hist',
                                  eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
            m.fit(X22b.iloc[tr].drop(columns=drop), y[tr], sample_weight=compute_sample_weight('balanced', y[tr]))
            l = LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced', random_state=SEED).fit(A[tr], y[tr])
            np.savez(p, va=va, xv=m.predict_proba(X22b.iloc[va].drop(columns=drop)), xt=m.predict_proba(X22b_te.drop(columns=drop)),
                     lv=l.predict_proba(A[va]), lt=l.predict_proba(A_te))
            z = np.load(p); print(f'  {tag}fold{k}: 학습 {time.time() - t:.0f}s')
        oof[va] = 0.7 * z['xv'] + 0.3 * z['lv']; te += (0.7 * z['xt'] + 0.3 * z['lt']) / len(folds)
    return oof, te

SCHEMES = {'① 공통 CV': (FOLDS_STD, 's'), '② 같은 프로필을 한 fold에': (FOLDS_GRP, 'g')}
C_PROBA = {name: run_c(folds, tag) for name, (folds, tag) in SCHEMES.items()}
for name, (folds, _) in SCHEMES.items():
    s = np.array([macro_f1(y[va], C_PROBA[name][0][va].argmax(1)) for _, va in folds])
    print(f'{name}: C CV {s.mean():.4f} ± {s.std():.4f}')

  sfold0: 학습 74s


  sfold1: 학습 78s


  sfold2: 학습 76s


  sfold3: 학습 79s


  sfold4: 학습 78s
① 공통 CV: C CV 0.4613 ± 0.0094
② 같은 프로필을 한 fold에: C CV 0.4641 ± 0.0170


#### 3) 쌍 안에서 라벨을 구분할 수 있는가
- 쌍에 속한 환자만 놓고, C의 확률로 세부 암종 라벨을 맞히는 AUC (0.5 = 구분 못 함, 0.5 미만 = 거꾸로 맞힘)

In [4]:
rows = []
for name, (folds, _) in SCHEMES.items():
    P = C_PROBA[name][0]
    for big, sub in PAIRS:
        a, b = ci[big], ci[sub]; m = (y == a) | (y == b)
        rows.append({'검증': name, '쌍': f'{big} vs {sub}', 'AUC': roc_auc_score((y[m] == b).astype(int), (P[:, b] / (P[:, a] + P[:, b]))[m])})
pd.DataFrame(rows).pivot(index='쌍', columns='검증', values='AUC').round(3)

검증,① 공통 CV,② 같은 프로필을 한 fold에
쌍,,
GBMLGG vs LGG,0.550,0.705
KIPAN vs KIRC,0.298,0.603


#### 4) 계층 분류: C vs C + 2단계 (두 검증 모두)

In [5]:
def hierarchical(P, P_te, folds):
    Q, Q_te = P.copy(), P_te.copy()
    for big, sub in PAIRS:
        a, b = ci[big], ci[sub]
        s, s_te = np.zeros(len(y)), np.zeros(len(test))
        for tr, va in folds:
            m = tr[(y[tr] == a) | (y[tr] == b)]                       # 학습 fold 안의 쌍 환자만
            t = pd.Series((y[m] == b).astype(int)).groupby(grp[m]).transform('max').to_numpy()   # 같은 프로필에 세부 암종 라벨이 있으면 1
            mdl = LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced').fit(A[m], t)
            s[va] = mdl.predict_proba(A[va])[:, 1]; s_te += mdl.predict_proba(A_te)[:, 1] / len(folds)
        G, G_te = P[:, a] + P[:, b], P_te[:, a] + P_te[:, b]
        Q[:, b], Q[:, a] = G * s, G * (1 - s)
        Q_te[:, b], Q_te[:, a] = G_te * s_te, G_te * (1 - s_te)
    return Q, Q_te

rows, per_class, H_PROBA = [], {}, {}
for name, (folds, _) in SCHEMES.items():
    P, P_te = C_PROBA[name]; Q, Q_te = hierarchical(P, P_te, folds); H_PROBA[name] = (Q, Q_te)
    fs = lambda X: np.array([macro_f1(y[va], X[va].argmax(1)) for _, va in folds])
    base, h = fs(P), fs(Q); d = h - base
    rows.append({'검증': name, 'C': base.mean(), '계층': h.mean(), 'C 대비': d.mean(), 'C보다 높은 fold': f'{(d > 0).sum()}/5'})
    f0, f1 = f1_score(y, P.argmax(1), average=None), f1_score(y, Q.argmax(1), average=None)
    rest = [i for c, i in ci.items() if c not in FOUR]
    per_class[name] = {**{c: f'{f0[ci[c]]:.3f} → {f1[ci[c]]:.3f}' for c in FOUR}, '나머지 22개 평균': f'{f0[rest].mean():.3f} → {f1[rest].mean():.3f}'}
print(pd.DataFrame(rows).set_index('검증').round(4))
print('\n암종별 F1 (C → 계층)'); pd.DataFrame(per_class)

                        C      계층    C 대비 C보다 높은 fold
검증                                                   
① 공통 CV            0.4613  0.4692  0.0079         5/5
② 같은 프로필을 한 fold에  0.4641  0.4607 -0.0033         0/5

암종별 F1 (C → 계층)


,① 공통 CV,② 같은 프로필을 한 fold에
KIPAN,0.211 → 0.190,0.317 → 0.245
KIRC,0.189 → 0.270,0.413 → 0.460
GBMLGG,0.336 → 0.370,0.386 → 0.354
LGG,0.462 → 0.573,0.572 → 0.577
나머지 22개 평균,0.492 → 0.492,0.470 → 0.469


## 결과 (Linux 실행, XGB 시드 42 하나)

**1) train 안의 같은 프로필·다른 라벨 행**
- 유전자 4,384칸이 모두 같은 행 묶음이 451개 (행 1,016개). 대부분이 **KIPAN + KIRC (272묶음)** 와 **GBMLGG + LGG (173묶음)**
- KIRC 환자의 82.6%는 같은 프로필의 KIPAN 행이 있고, LGG 환자의 76.9%는 같은 프로필의 GBMLGG 행이 있음 → 같은 환자가 두 라벨로 한 번씩 들어 있는 구조
- 공통 CV에서는 이 묶음의 80.7%가 서로 다른 fold로 갈라짐

**2) 쌍 안에서 라벨 구분 (C의 확률, AUC)**

| 쌍 | ① 공통 CV | ② 같은 프로필을 한 fold에 |
| :-- | :-: | :-: |
| KIPAN vs KIRC | 0.298 | 0.603 |
| GBMLGG vs LGG | 0.550 | 0.705 |

- 공통 CV에서 KIPAN vs KIRC의 AUC가 0.5보다 훨씬 낮음 = **거꾸로 맞힘**
  - 같은 환자의 한 행(예: KIRC)이 학습 fold에, 다른 행(KIPAN)이 검증 fold에 있으면 모델은 학습에서 본 라벨(KIRC)을 내고, 검증 정답은 반대 라벨(KIPAN)
  - 그래서 공통 CV의 이 4개 암종 F1(0.19~0.46)은 실제 구분력보다 낮게 나온 값

**3) 계층 분류**

| 검증 | C | 계층 | C 대비 | C보다 높은 fold |
| :-- | :-: | :-: | :-: | :-: |
| ① 공통 CV | 0.4613 | 0.4692 | +0.0079 | 5/5 |
| ② 같은 프로필을 한 fold에 | 0.4641 | 0.4607 | −0.0033 | 0/5 |

- ①에서는 오르지만 ②에서는 5개 fold 모두 하락
- ①의 상승은 "거꾸로 맞히던 것"이 줄어든 효과 (KIRC 0.19 → 0.27, LGG 0.46 → 0.57) → 같은 프로필이 갈라진 fold 구조에서만 생기는 이득
- 같은 프로필을 한 fold에 두면 C가 이미 4개 암종을 더 잘 맞히고(KIRC 0.41, LGG 0.57), 2단계를 붙이면 KIPAN(0.32 → 0.25)·GBMLGG(0.39 → 0.35)가 떨어져 손해
- 나머지 22개 암종은 두 검증 모두 변화 없음

## 결론
- **H29 기각**: 계층 분류는 공통 CV에서만 오르고(+0.0079), 같은 프로필을 한 fold에 묶은 검증에서는 내려감(−0.0033, 0/5) → 두 검증 모두에서 올라야 한다는 기준 미달, 제출 파일 없음
- 공통 CV의 상승은 모델이 좋아진 것이 아니라 **검증 구조(같은 환자가 두 fold에 나뉨)에서 생긴 착시**로 판단
- 발견 (발표·한계에 쓸 내용)
  - KIPAN↔KIRC, GBMLGG↔LGG는 같은 환자가 두 라벨로 들어 있어, 변이 정보만으로는 어느 라벨인지 정할 수 없음 → 이 4개 암종의 오분류는 모델 한계가 아니라 **라벨 구조의 한계**
  - 공통 CV는 이 4개 암종의 점수를 실제보다 낮게 매김 (같은 환자가 갈라져 거꾸로 맞힘) → 실험 간 비교에는 영향이 같아서 지금까지의 채택 판단은 그대로 유효
- 하지 않은 것: test 환자를 train의 같은 프로필 행과 맞춰 보거나, 그 라벨로 예측을 고치는 후처리 (팀 규칙)
